In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix, vstack
import faiss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
from sklearn.model_selection import KFold


In [ ]:
def build_sparse_matrix(ratings_df):
    user_ids = ratings_df["userId"].unique()
    movie_ids = ratings_df["movieId"].unique()

    user_map = {uid: idx for idx, uid in enumerate(user_ids)}
    movie_map = {mid: idx for idx, mid in enumerate(movie_ids)}
    reverse_movie_map = {idx: mid for mid, idx in movie_map.items()}

    row = ratings_df["userId"].map(user_map)
    col = ratings_df["movieId"].map(movie_map)
    data = ratings_df["rating"]

    sparse_matrix = csr_matrix((data, (row, col)), shape=(len(user_map), len(movie_map)))
    return sparse_matrix, user_map, movie_map, reverse_movie_map

In [ ]:
from sklearn.preprocessing import normalize
import faiss
import numpy as np

def build_faiss_index_batched(sparse_matrix):
    dense = normalize(sparse_matrix.toarray().astype("float32"))
    d = dense.shape[1]

    quantizer = faiss.IndexFlatIP(d)
    index = faiss.IndexIVFFlat(quantizer, d, nlist=10, metric=faiss.METRIC_INNER_PRODUCT)
    index.train(dense)
    index.add(dense)
    return index

In [2]:
# Load datasets
ratings_df = pd.read_csv("../data/ratings.csv")
movies_df = pd.read_csv("../data/movies.csv")


In [ ]:
def create_per_user_split(ratings_df, test_ratio=0.2, min_ratings=5, seed=42):
    train_rows, test_rows = [], []

    for user_id, user_ratings in ratings_df.groupby("userId"):
        if len(user_ratings) < min_ratings:
            train_rows.append(user_ratings)
            continue

        shuffled = user_ratings.sample(frac=1.0, random_state=seed)
        split = int(len(shuffled) * test_ratio)
        test_rows.append(shuffled.iloc[:split])
        train_rows.append(shuffled.iloc[split:])

    train_df = pd.concat(train_rows)
    test_df = pd.concat(test_rows)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [14]:
def sample_users(ratings_df, min_ratings=10, sample_size=5000, seed=42):
    users = ratings_df['userId'].value_counts()
    eligible_users = users[users >= min_ratings].index
    np.random.seed(seed)
    sampled_users = np.random.choice(eligible_users, size=sample_size, replace=False)
    return ratings_df[ratings_df['userId'].isin(sampled_users)].copy()

In [15]:
# 1. Sample 5,000 users
ratings_sampled = sample_users(ratings_df, sample_size=5000)

# 2. Per-user split
train_df, test_df = create_per_user_split(ratings_sampled, test_ratio=0.2)

# 3. Build sparse matrix + index
sparse_matrix, user_map, movie_map, reverse_movie_map = build_sparse_matrix(train_df)
faiss_index = build_faiss_index_batched(sparse_matrix)

# 4. Filter test to known users/movies
test_df = test_df[test_df["userId"].isin(user_map) & test_df["movieId"].isin(movie_map)]

# 5. Evaluate
rmse = evaluate_rmse(test_df, sparse_matrix, faiss_index, user_map, movie_map, k=20)
print(f"Sampled users RMSE (n=5000): {rmse:.4f}")


Evaluating: 100%|██████████| 148703/148703 [07:14<00:00, 342.62it/s]


Sampled users RMSE (n=5000): 1.0077


In [16]:
def sample_users_for_cv(ratings_df, min_ratings=10, sample_size=5000, seed=42):
    users = ratings_df['userId'].value_counts()
    eligible = users[users >= min_ratings].index
    np.random.seed(seed)
    sampled_users = np.random.choice(eligible, size=sample_size, replace=False)
    return sampled_users

In [17]:
from sklearn.model_selection import KFold

def cross_val_user_stratified_sampled(ratings_df, movies_df=None, k_folds=5, k=20, sample_size=5000):
    sampled_users = sample_users_for_cv(ratings_df, sample_size=sample_size)
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    fold_rmses = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(sampled_users), 1):
        print(f"\nFold {fold}/{k_folds}")
        train_users = sampled_users[train_idx]
        test_users = sampled_users[test_idx]

        # Subset ratings
        train_ratings_all = ratings_df[ratings_df["userId"].isin(train_users)].copy()
        test_ratings_all = ratings_df[ratings_df["userId"].isin(test_users)].copy()

        # Per-user 80/20 split for test users
        train_rows, test_rows = [], []
        for uid in test_users:
            user_data = test_ratings_all[test_ratings_all["userId"] == uid]
            if len(user_data) < 5:
                continue
            user_data = user_data.sample(frac=1.0, random_state=42)
            split = int(len(user_data) * 0.2)
            test_rows.append(user_data.iloc[:split])
            train_rows.append(user_data.iloc[split:])
        test_df = pd.concat(test_rows)
        test_train_df = pd.concat(train_rows)

        # Final training set: other folds + 80% of this fold
        final_train_df = pd.concat([train_ratings_all, test_train_df])

        # Build model
        sparse_matrix, user_map, movie_map, reverse_movie_map = build_sparse_matrix(final_train_df)
        faiss_index = build_faiss_index_batched(sparse_matrix)

        # Filter test
        test_df = test_df[test_df["userId"].isin(user_map) & test_df["movieId"].isin(movie_map)]

        # Evaluate
        rmse = evaluate_rmse(test_df, sparse_matrix, faiss_index, user_map, movie_map, k=k)
        print(f"  Fold RMSE: {rmse:.4f}")
        fold_rmses.append(rmse)

    print(f"\nCross-Validated Sampled RMSE: {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")
    return fold_rmses

In [18]:
rmse_scores = cross_val_user_stratified_sampled(ratings_df, k_folds=5, sample_size=5000)


Fold 1/5


Evaluating: 100%|██████████| 29222/29222 [01:28<00:00, 330.18it/s]


  Fold RMSE: 1.0179

Fold 2/5


Evaluating: 100%|██████████| 30326/30326 [01:34<00:00, 319.97it/s]


  Fold RMSE: 0.9933

Fold 3/5


...

  Fold RMSE: 0.9921

Fold 4/5


Evaluating: 100%|██████████| 28291/28291 [01:20<00:00, 352.09it/s]


  Fold RMSE: 1.0104

Fold 5/5


Evaluating: 100%|██████████| 32609/32609 [01:25<00:00, 383.09it/s]

  Fold RMSE: 0.9945

Cross-Validated Sampled RMSE: 1.0016 ± 0.0105
